# Building upon the model and evaluate method used in InsectNet
https://academic.oup.com/pnasnexus/article/4/1/pgae575/7933354?login=false

## Pipeline Overview

1. **Weather classification** from EXIF shutter speed (per image, no training needed)
2. **Define ROI** via green marker auto-detection or manual selection
3. **Build median background** from uniform sample (memory-efficient)
4. **Per-frame detection**: background subtraction (+ optional MIE) then shape/texture filtering
5. **Static detection removal**: reject positions appearing in too many frames
6. **InsectNet classification**: crop detections, classify, map to broad category
7. **Near-flower check**: verify detection overlaps with ROI zone
8. **CSV output**: one row per image wih all fields required by collaborator

- All tunable parameters are in `DEFAULT_CONFIG` dict — override per run without touching code — override per run without touching code
- Background built from uniform sample of 200 frames instead of all frames (avoids memory crash on 12k images)
- Weather classified from EXIF ExposureTime (shutter speed) — no training needed
- Pollinator broad class mapped from InsectNet Order
- CSV output with all fields required by Maria
- `use_mie=True` enables Motion-Informed Enhancement (Bjerge et al. 2023): embeds cumulative motion into Blue channel, reducing wind/grass false positives

### Imports

In [11]:
import sys
import csv
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image as PILImage
from PIL.ExifTags import TAGS

sys.path.insert(0, "/workspace/Engineering Thesis/Pollinator")
from evaluate import evaluate

PROJECT_DIR = Path.cwd()

### Configuration

All tunable parameters are here. Override any of them when calling `main()` without touching the rest of the code.

Parameters marked **TUNE** are the ones most likely to need adjustment per dataset or plot.

In [12]:
DEFAULT_CONFIG = {

    # Background computation
    # Number of frames to sample for median background.
    # Original loaded ALL frames (crashes at 12k images).
    # 200 uniformly-spaced frames uses ~320MB instead of ~60GB.
    "background_sample_size":    50,

    # Background subtraction
    # How much darker than the median a pixel must be to count as foreground.
    # Lower = more sensitive (more detections, more noise).
    # Higher = less sensitive (fewer detections, fewer false positives). TUNE
    "darker_threshold":          55,

    # Contour filtering
    # Minimum pixel area of a contour to be considered a visitor. TUNE
    "min_contour_area":          800,
    # Aspect ratio limit: max(w,h)/min(w,h) must be below this (rejects grass stems).
    "max_aspect_ratio":          2,
    # Minimum grayscale std dev inside contour (insects have texture, soil does not). TUNE
    "min_texture":               50,


    # Static detection removal
    # Detections within this pixel radius are treated as the same position.
    "static_dist":               80,
    # Reject a position if it appears in more than this many frames.
    # Was 2 (too aggressive). Bumblebees stay max ~5 frames; set to 10 to be safe. TUNE
    "static_max_frames":         10,

    # Crop padding
    "padding":                   50,

    
    # Rolling background window
    # 0 = use global median background (default)
    # N > 0 = use median of the last N frames — adapts to slow lighting changes
    # and reduces false positives from grass that moves consistently.
    # NOTE: insects staying > N frames may fade into background. Use N >= 20.
    "rolling_window": 0,   # TUNE — try 20 for windy plots

    # Motion-Informed Enhancement (MIE) — Bjerge et al. 2023
    # Replaces Blue channel with cumulative frame-to-frame motion.
    # Amplifies insects (sudden appearance) and suppresses wind/grass (oscillation).
    # Recommended: use together with rolling_window for best false positive reduction.
    "use_mie": False,      # TUNE — set True to enable MIE

    # Marker / ROI
    "marker_hue":                (45, 75),
    "marker_sat_min":            200,
    "marker_val_min":            100,
    "marker_min_area":           200,
    "marker_zone_radius":        800,

    # Weather classification from EXIF shutter speed
    # Camera: Wingscapes TLCAM PRO, fixed aperture f/2.8, auto exposure.
    # Sunny  = fast shutter, denominator > threshold (e.g. 1/1249 = 1249)
    # Cloudy = slow shutter, denominator <= threshold
    # IMPORTANT: verify this threshold against actual cloudy images. TUNE
    "sunny_shutter_threshold":   400,

    # Near-flower detection
    # Fraction of insect bbox overlapping ROI zone to count as near marked flower.
    "near_flower_iou_threshold": 0.1,

    # Pollinator class mapping
    # Maps InsectNet taxonomic Order to broad category for Maria's CSV.
    # Broad pollinator categories — anything not listed maps to "other"
    "pollinator_class_map": {
        "Hymenoptera": "bumblebee",
        "Diptera":     "fly",
        "Lepidoptera": "butterfly",
    },
}

CROP_DIR = Path("all_crops") 
CROP_DIR.mkdir(exist_ok=True)

CSV_FIELDS_DEBUG = [
    "image_name", "datetime", "camera_name",
    "shutter_speed", "weather",
    "pollinator_detected", "pollinator_type",
    "scientific_name", "common_name",
    "order", "family",
    "insectnet_confirmed", "confidence", "energy_score",
    "near_marked_flower",
    "detection_scope",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h",
]

CSV_FIELDS_MARIA = [
    "image_name", "datetime", "camera_name",
    "shutter_speed", "weather",
    "pollinator_detected", "pollinator_type",
    "scientific_name",
    "common_name",
    "order",
    "confidence", "energy_score",
    "near_marked_flower",
    "detection_scope",
]

### Helper functions

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════

def load_model():
    """Load InsectNet RegNet weights from model.pth."""
    weights = torch.load(
        PROJECT_DIR / "model.pth",
        map_location=torch.device("cpu"),
        weights_only=False,
    )["model"]
    model = torchvision.models.regnet_y_32gf()
    model.fc = torch.nn.Linear(3712, 2526)
    model.load_state_dict(weights, strict=True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    model.eval()
    return model


def load_model_and_classes():
    """Step 6: Load InsectNet model and class metadata."""
    print("Loading model...")
    model = load_model()
    cmn_df = pd.read_csv(PROJECT_DIR / "classes.csv")
    class_txt_path = str(PROJECT_DIR / "classes.txt")
    print("Model loaded.\n")
    return model, cmn_df, class_txt_path


# ══════════════════════════════════════════════════════════════════
# ROI / ZONE
# ══════════════════════════════════════════════════════════════════

def find_marker(image, cfg):
    """Find the green marker clip in the image. Returns centroid (cx, cy) or None."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(
        hsv,
        (cfg["marker_hue"][0], cfg["marker_sat_min"], cfg["marker_val_min"]),
        (cfg["marker_hue"][1], 255, 255),
    )
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clusters = [c for c in contours if cv2.contourArea(c) > cfg["marker_min_area"]]
    if not clusters:
        return None
    h_img, w_img = image.shape[:2]
    cx_img, cy_img = w_img // 2, h_img // 2

    def dist_to_center(c):
        M = cv2.moments(c)
        if M["m00"] == 0:
            return float("inf")
        return (int(M["m10"]/M["m00"]) - cx_img)**2 + (int(M["m01"]/M["m00"]) - cy_img)**2

    best = min(clusters, key=dist_to_center)
    M = cv2.moments(best)
    return int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])


def build_marker_zone(image, marker, cfg):
    """Create a circular binary mask (zone) around the marker position."""
    h_img, w_img = image.shape[:2]
    zone = np.zeros((h_img, w_img), dtype=np.uint8)
    cv2.circle(zone, marker, cfg["marker_zone_radius"], 255, -1)
    return zone


def select_roi(image_path):
    """Open the first image and let user draw a rectangle as the watching zone."""
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    scale = min(1.0, 1200 / w)
    display = cv2.resize(img, (int(w * scale), int(h * scale)))
    print("Draw a rectangle around the flower area, then press ENTER or SPACE.")
    roi = cv2.selectROI("Select flower region", display, showCrosshair=True)
    cv2.destroyAllWindows()
    x, y, rw, rh = [int(v / scale) for v in roi]
    if rw == 0 or rh == 0:
        return None
    zone = np.zeros((h, w), dtype=np.uint8)
    zone[y:y+rh, x:x+rw] = 255
    print(f"ROI selected: x={x} y={y} w={rw} h={rh}")
    return zone


def setup_roi(paths, cfg, manual_roi):
    """Step 1: Define ROI from green marker or manual selection."""
    first_img = cv2.imread(str(paths[0]))
    if manual_roi:
        zone = select_roi(str(paths[0]))
        if zone is None:
            raise ValueError("No region selected.")
        return zone, None
    marker = find_marker(first_img, cfg)
    if marker is None:
        raise ValueError("No green marker found. Use manual_roi=True to draw a region.")
    print(f"Marker found at ({marker[0]}, {marker[1]})")
    zone = build_marker_zone(first_img, marker, cfg)
    pct = 100 * np.count_nonzero(zone) / zone.size
    print(f"Watching zone covers {pct:.1f}% of image")
    return zone, marker


# ══════════════════════════════════════════════════════════════════
# BACKGROUND
# ══════════════════════════════════════════════════════════════════

def build_background(paths, cfg):
    """
    Step 2: Compute median background from a uniform sample of frames.

    Uses cfg['background_sample_size'] frames evenly spaced across the sequence.
    Avoids loading all 12,000 frames into memory (~60GB) while producing
    a background equivalent to the full-dataset median.
    Per-pixel median naturally excludes transient objects like insects (~4% of frames).
    """
    n = cfg["background_sample_size"]
    step = max(1, len(paths) // n)
    sampled = list(paths)[::step][:n]
    frames = [cv2.imread(str(p)) for p in sampled]
    frames = [f for f in frames if f is not None]
    if not frames:
        return None
    print(f"Background computed from {len(frames)} sampled frames (of {len(paths)} total)")
    return np.median(frames, axis=0).astype(np.uint8)


# ══════════════════════════════════════════════════════════════════
# DETECTION (background subtraction + contour filtering)
# ══════════════════════════════════════════════════════════════════

def detect_visitor(image, background, zone, cfg):
    """
    Find candidate insect regions in one frame by comparing to background.

    Strategy:
    1. Subtract background — pixels darker than median are potential foreground
    2. Exclude green vegetation (grass, leaves)
    3. Restrict to ROI zone
    4. Filter contours by area, aspect ratio, and texture
       - Too small  → noise
       - Too elongated → grass stems
       - No texture → shadow / soil

    Returns list of bounding boxes [(x, y, w, h), ...].
    Note: these are candidates only; InsectNet classifies them in Step 7.
    """
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.int16)
    gray_bg  = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY).astype(np.int16)
    darker   = np.clip(gray_bg - gray_img, 0, 255).astype(np.uint8)
    darker   = cv2.GaussianBlur(darker, (7, 7), 0)
    _, mask  = cv2.threshold(darker, cfg["darker_threshold"], 255, cv2.THRESH_BINARY)
    mask     = cv2.bitwise_and(mask, zone)

    # Exclude green vegetation
    hsv   = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, (25, 40, 40), (95, 255, 255))
    mask  = cv2.bitwise_and(mask, cv2.bitwise_not(green))

    # Morphological cleanup — remove speckles, fill gaps
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < cfg["min_contour_area"]:
            continue
        x, y, w, h = cv2.boundingRect(c)
        if max(w, h) / max(min(w, h), 1) > cfg["max_aspect_ratio"]:
            continue
                
        if cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)[y:y+h, x:x+w].std() < cfg["min_texture"]:
            continue
        valid.append((area, cv2.boundingRect(c)))

    valid.sort(key=lambda v: v[0], reverse=True)
    return [bbox for _, bbox in valid]


# ══════════════════════════════════════════════════════════════════
# MOTION-INFORMED ENHANCEMENT (MIE)
# ══════════════════════════════════════════════════════════════════

def motion_enhanced(curr_img, prev_img, prev_diff):
    """
    Motion-Informed Enhancement (MIE) — Bjerge et al. 2023
    Sensors 23(16): 7242. https://doi.org/10.3390/s23167242

    Embeds cumulative motion information into the Blue channel of the image
    before passing it to the insect detector.

    How it works:
      - Compute per-pixel absolute difference between current and previous frame
      - Add to the previous frame's difference (cumulative motion)
      - Replace Blue channel of the previous frame with this cumulative motion map

    Why it reduces false positives from wind/grass:
      - Grass oscillates back and forth → differences cancel out over two frames
      - An insect appears suddenly → difference accumulates → Blue channel is bright
    

    Args:
        curr_img:  current frame (BGR)
        prev_img:  previous frame (BGR)
        prev_diff: absolute difference from the previous pair of frames (grayscale)
                   pass None for the first frame

    Returns:
        enhanced:  MIE image ready 
        curr_diff: current frame difference (to pass as prev_diff next iteration)
    """
    curr_gray = cv2.GaussianBlur(cv2.cvtColor(curr_img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    prev_gray = cv2.GaussianBlur(cv2.cvtColor(prev_img, cv2.COLOR_BGR2GRAY), (5, 5), 0)

    curr_diff = cv2.absdiff(prev_gray, curr_gray)
    sum_diff  = cv2.add(prev_diff, curr_diff) if prev_diff is not None else curr_diff

    # Embed motion into Blue channel of previous frame
    enhanced         = prev_img.copy()
    enhanced[:, :, 0] = prev_img[:, :, 0] // 2 + prev_img[:, :, 2] // 2  # R+B blend
    enhanced[:, :, 2] = sum_diff  # Blue = cumulative motion intensity

    return enhanced, curr_diff

def detect_all_frames(paths, background, zone, cfg, debug=False, debug_dir=None):
    """
    Step 4: Run background subtraction (with optional MIE) on every frame.

    If cfg['use_mie'] is True, applies Motion-Informed Enhancement (Bjerge et al. 2023)
    before detection: the Blue channel of each frame is replaced with cumulative
    motion information, amplifying insects and suppressing wind/grass oscillation.

    If cfg['rolling_window'] > 0, uses a rolling median background instead of
    the global median, further reducing slow lighting-change false positives.
    """
    print("Detecting visitors...")
    all_detections = {}
    window    = cfg.get("rolling_window", 0)
    use_mie   = cfg.get("use_mie", False)

    frames_cache = []  # for rolling background
    prev_img     = None  # for MIE
    prev_diff    = None  # for MIE

    for path in paths:
        image = cv2.imread(str(path))
        if image is None:
            continue

        # Choose background: rolling (last N frames) or global median
        if window > 0 and len(frames_cache) >= 2:
            # Rolling background: grass that moves consistently across recent
            # frames is absorbed into the background and will not be detected.
            bg = np.median(frames_cache[-window:], axis=0).astype(np.uint8)
        else:
            bg = background

        # Apply MIE: replace Blue channel with cumulative motion map.
        # Insects appear suddenly → motion accumulates → detected.
        # Grass oscillates → motion cancels → not detected.
        if use_mie and prev_img is not None:
            img_to_detect, prev_diff = motion_enhanced(image, prev_img, prev_diff)
        else:
            img_to_detect = image
            if use_mie and prev_img is None:
                prev_diff = None  # reset for first frame

        if debug and debug_dir:
            save_debug_images(path.stem, img_to_detect, bg, zone, None, debug_dir, cfg)

        all_detections[str(path)] = detect_visitor(img_to_detect, bg, zone, cfg)

        # Update histories
        prev_img = image  # always store original for MIE
        frames_cache.append(image)
        if window > 0 and len(frames_cache) > window:
            frames_cache.pop(0)

    return all_detections


def filter_static_detections(all_detections, cfg):
    """
    Step 5: Remove detections at the same position across too many frames.

    Real visitors appear in a few consecutive frames then leave.
    Soil patches, fixed shadows, and other static artefacts persist across
    many frames and are rejected here.

    static_max_frames is set to 15 (was 2 in original):
    bumblebees can stay on a flower for several minutes (~5 frames at 1-min intervals).
    """
    all_centers = [
        (x + w//2, y + h//2, path)
        for path, bboxes in all_detections.items()
        for x, y, w, h in bboxes
    ]

    static_positions = set()
    for cx, cy, _ in all_centers:
        frames_nearby = {p for cx2, cy2, p in all_centers
                         if ((cx-cx2)**2 + (cy-cy2)**2)**0.5 < cfg["static_dist"]}
        if len(frames_nearby) > cfg["static_max_frames"]:
            static_positions.add((cx, cy))

    if static_positions:
        print(f"Filtered {len(static_positions)} static position(s) (appeared in >{cfg['static_max_frames']} frames)")

    filtered = {}
    for path, bboxes in all_detections.items():
        filtered[path] = [
            (x, y, w, h) for x, y, w, h in bboxes
            if not any(((x+w//2-sx)**2 + (y+h//2-sy)**2)**0.5 < cfg["static_dist"]
                       for sx, sy in static_positions)
        ]
    return filtered


def crop_with_padding(image, bbox, cfg):
    """Crop a bounding box from the image with extra padding around it."""
    x, y, w, h = bbox
    h_img, w_img = image.shape[:2]
    pad = cfg["padding"]
    return image[max(0,y-pad):min(h_img,y+h+pad), max(0,x-pad):min(w_img,x+w+pad)]


# ══════════════════════════════════════════════════════════════════
# WEATHER (EXIF)
# ══════════════════════════════════════════════════════════════════

def get_exif_metadata(image_path, cfg):
    """
    Step 3: Extract metadata and classify weather from EXIF shutter speed.

    Camera: Wingscapes TLCAM PRO — fixed aperture f/2.8, auto exposure.
    Shutter speed reflects ambient light:
      sunny  → fast shutter → denominator > sunny_shutter_threshold
      cloudy → slow shutter → denominator <= sunny_shutter_threshold

    NOTE: verify threshold against actual cloudy images before use.
    """
    metadata = {"datetime": "", "camera_name": "", "shutter_speed": "", "weather": "unknown"}
    try:
        exif_data = PILImage.open(image_path)._getexif()
        if exif_data is None:
            return metadata
        tag_map = {TAGS.get(k, k): v for k, v in exif_data.items()}
        metadata["datetime"]    = str(tag_map.get("DateTimeOriginal", ""))
        metadata["camera_name"] = str(tag_map.get("Model", ""))
        exposure = exif_data.get(33434)  # ExposureTime EXIF tag
        if exposure:
            denom = exposure[1] if isinstance(exposure, tuple) else int(1 / exposure)
            metadata["shutter_speed"] = f"1/{denom}"
            metadata["weather"] = "sunny" if denom > cfg["sunny_shutter_threshold"] else "cloudy"
    except Exception:
        pass
    return metadata


# ══════════════════════════════════════════════════════════════════
# CLASSIFICATION HELPERS
# ══════════════════════════════════════════════════════════════════

def map_to_broad_class(order, cfg):
    """Map InsectNet taxonomic Order to broad pollinator category (from cfg)."""
    return cfg["pollinator_class_map"].get(order, "other")


def is_near_marked_flower(bbox, zone, cfg):
    """
    Returns True if the insect bounding box overlaps sufficiently with the ROI zone.
    Uses near_flower_iou_threshold as the minimum overlap fraction.
    """
    x, y, w, h = bbox
    roi_crop = zone[y:y+h, x:x+w]
    if roi_crop.size == 0:
        return False
    return np.count_nonzero(roi_crop) / roi_crop.size > cfg["near_flower_iou_threshold"]


# ══════════════════════════════════════════════════════════════════
# CSV
# ══════════════════════════════════════════════════════════════════

def init_csv(output_path, fields):
    """Create CSV file with headers. Overwrites existing file."""
    with open(output_path, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=fields).writeheader()


def write_csv_row(output_path, row_dict, fields):
    """Append one row to the CSV. Missing fields are written as empty string."""
    with open(output_path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=fields).writerow(
            {k: row_dict.get(k, "") for k in fields}
        )


# ══════════════════════════════════════════════════════════════════
# DEBUG IMAGES
# ══════════════════════════════════════════════════════════════════

def save_debug_images(name, image, background, zone, marker, debug_dir, cfg):
    """Save intermediate debug images for visual inspection of the pipeline."""
    # 1. Zone overlay — shows the ROI (purple tint) and marker (green dot)
    overlay = image.copy()
    overlay[zone > 0] = overlay[zone > 0] // 2 + np.array([128, 0, 128], dtype=np.uint8)
    if marker:
        cv2.circle(overlay, marker, 15, (0, 255, 0), -1)
    cv2.imwrite(str(debug_dir / f"{name}_1_zone.jpg"), overlay)

    # 2. Difference mask — white = darker than background (potential foreground)
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.int16)
    gray_bg  = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY).astype(np.int16)
    darker   = np.clip(gray_bg - gray_img, 0, 255).astype(np.uint8)
    darker   = cv2.GaussianBlur(darker, (7, 7), 0)
    _, mask  = cv2.threshold(darker, cfg["darker_threshold"], 255, cv2.THRESH_BINARY)
    mask     = cv2.bitwise_and(mask, zone)
    hsv      = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask     = cv2.bitwise_and(mask, cv2.bitwise_not(cv2.inRange(hsv, (25,40,40), (95,255,255))))
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask     = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel), cv2.MORPH_CLOSE, kernel)
    cv2.imwrite(str(debug_dir / f"{name}_2_diff.jpg"), mask)

    # 3. Contours — green = passes all filters, red = rejected
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay2 = image.copy()
    for c in contours:
        area = cv2.contourArea(c)
        if area < 100:
            continue
        x, y, w, h = cv2.boundingRect(c)
        roi = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)[y:y+h, x:x+w]
        passes = (area >= cfg["min_contour_area"]
                  and max(w,h)/max(min(w,h),1) <= cfg["max_aspect_ratio"]
                  and roi.std() >= cfg["min_texture"])
        color = (0, 255, 0) if passes else (0, 0, 255)
        cv2.rectangle(overlay2, (x,y), (x+w,y+h), color, 2)
        cv2.putText(overlay2, f"a={int(area)} t={roi.std():.0f}", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    cv2.imwrite(str(debug_dir / f"{name}_3_contours.jpg"), overlay2)


### Pipeline functions & Main

In [14]:
def classify_and_write(paths, filtered_full, zone, model, cmn_df,
                       class_txt_path, output_csv, cfg, csv_fields,
                       debug=False, debug_dir=None):
    """
    Step 7: Classify all detections from full image and write to CSV.

    Each detection is checked against the ROI zone to set detection_scope:
      "roi"         — bbox overlaps with the marked flower zone
      "outside_roi" — bbox is elsewhere in the image
    Running one full-image detection pass is simpler than two separate passes.
    """
    init_csv(output_csv, csv_fields)

    for path in paths:
        path_str = str(path)
        image = cv2.imread(path_str)
        if image is None:
            continue

        # Classify weather from EXIF shutter speed
        metadata   = get_exif_metadata(path_str, cfg)
        detections = filtered_full.get(path_str, [])

        if not detections:
            print(f"{Path(path).name}: no visitor detected ({metadata['weather']})")
            write_csv_row(output_csv,
                {"image_name": Path(path).name, **metadata, "pollinator_detected": "no"},
                csv_fields)
            continue

        for i, bbox in enumerate(detections):
            crop     = crop_with_padding(image, bbox, cfg)
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            if debug and debug_dir:
                cv2.imwrite(str(debug_dir / f"{Path(path).stem}_4_crop_{i}.jpg"), crop)

            cv2.imwrite(str(CROP_DIR / f"{Path(path).stem}_crop_{i}.jpg"), crop)

            sci, cmn, order, family, role, confirmed, other, confidence, energy = evaluate(
                model, crop_rgb, cmn_df, class_txt_path
            )

            x, y, w, h = bbox

            # Determine scope: is this detection inside the ROI zone?
            in_roi = is_near_marked_flower(bbox, zone, cfg)
            scope  = "roi" if in_roi else "outside_roi"

            # Filter 1: OOD — InsectNet not confident
            if not confirmed:
                print(f"  {Path(path).name} [{scope}]: skipped — OOD (energy={energy:.2f})")
                write_csv_row(output_csv, {
                    "image_name": Path(path).name, **metadata,
                    "pollinator_detected": "uncertain",
                    "insectnet_confirmed": "False",
                    "scientific_name":     sci,
                    "detection_scope":     scope,
                    "near_marked_flower":  str(in_roi),
                    "confidence":          f"{confidence:.4f}",
                    "energy_score":        f"{energy:.4f}",
                    "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h,
                }, csv_fields)
                continue

            # Filter 2: role — only keep ecologically relevant roles
            if role not in ["Pollinator", "Predator", "Parasitoid"]:
                print(f"  {Path(path).name} [{scope}]: skipped — role={role}")
                continue

            broad_class = map_to_broad_class(order, cfg)

            print(f"{Path(path).name} [{scope}]")
            print(f"  Weather:         {metadata['weather']} ({metadata['shutter_speed']})")
            print(f"  Broad class:     {broad_class}")
            print(f"  Scientific Name: {sci}")
            print(f"  Common Name:     {cmn}")
            print(f"  Order/Family:    {order} / {family}")
            print(f"  Confirmed:       {confirmed}  |  Confidence: {confidence:.1%}  |  Energy: {energy:.2f}")
            print(f"  Near flower:     {in_roi}  |  Scope: {scope}")
            if other:
                print(f"  Other plausible: {other}")
            print()

            write_csv_row(output_csv, {
                "image_name": Path(path).name, **metadata,
                "pollinator_detected": "yes",
                "pollinator_type":     broad_class,
                "scientific_name":     sci,
                "common_name":         cmn,
                "order":               order,
                "family":              family,
                "insectnet_confirmed": str(confirmed),
                "near_marked_flower":  str(in_roi),
                "detection_scope":     scope,
                "confidence":          f"{confidence:.4f}",
                "energy_score":        f"{energy:.4f}",
                "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h,
            }, csv_fields)


def main(image_dir, output_csv="results.csv", debug=False,
         manual_roi=False, config=None, debug_csv=False):
    """
    Main pipeline — orchestrates all steps.

    Args:
        image_dir:  path to folder containing images for one plot
        output_csv: path to output CSV file
        debug:      save intermediate debug images to debug/ folder
        manual_roi: manually draw ROI instead of using green marker
        config:     dict of parameter overrides merged with DEFAULT_CONFIG
                    e.g. config={"sunny_shutter_threshold": 600}
        debug_csv:  False = clean CSV for Maria (default)
                    True  = full debug CSV with bbox, confidence, energy
    """
    cfg        = {**DEFAULT_CONFIG, **(config or {})}
    csv_fields = CSV_FIELDS_DEBUG if debug_csv else CSV_FIELDS_MARIA

    image_dir = Path(image_dir)
    paths = sorted(image_dir.glob("*.JPG")) + sorted(image_dir.glob("*.jpg"))
    if not paths:
        print(f"No images found in {image_dir}")
        return
    print(f"Found {len(paths)} images")

    debug_dir = PROJECT_DIR / "debug"
    if debug:
        debug_dir.mkdir(exist_ok=True)
        print(f"Debug images will be saved to {debug_dir}/")

    #  Define ROI
    zone, _ = setup_roi(paths, cfg, manual_roi)

    # Build median background (memory-efficient uniform sample)
    background = build_background(paths, cfg)
    if background is None:
        print("Failed to build background.")
        return

    #  Detect visitor candidates across the full image
    # Each detection will be labelled "roi" or "outside_roi" in Step 7
    h_img, w_img = cv2.imread(str(paths[0])).shape[:2]
    full_zone      = np.ones((h_img, w_img), dtype=np.uint8) * 255
    all_detections = detect_all_frames(paths, background, full_zone, cfg, debug, debug_dir)

    # Remove static detections (soil, shadows, fixed artefacts)
    filtered = filter_static_detections(all_detections, cfg)

    # Load InsectNet model
    model, cmn_df, class_txt_path = load_model_and_classes()

    # Classify candidates + write CSV (scope determined per bbox)
    classify_and_write(
        paths, filtered, zone, model, cmn_df,
        class_txt_path, output_csv, cfg, csv_fields,
        debug, debug_dir
    )
    print(f"\nDone. Results saved to {output_csv}")


### Run

Use default config, or override specific parameters via `config={}`. Only override what
need to change.

In [15]:
# Default run
main(
    image_dir  = PROJECT_DIR / "Insects_images",
    output_csv = "results.csv",
    debug      = True,
    manual_roi = False,
    config = {
        "marker_hue":      (80, 100), 
        "marker_sat_min":  100,        
        "marker_val_min":  80,
        "marker_min_area": 50,      
    }
)

# Example: override specific parameters after tuning
# main(
#     image_dir  = PROJECT_DIR / "images",
#     output_csv = "results.csv",
#     config = {
#         "sunny_shutter_threshold": 600,   # adjust after checking cloudy images
#         "static_max_frames":       10,    # tighten if too many false positives
#         "darker_threshold":        30,    # lower = more sensitive
#         "background_sample_size":  100,   # reduce if memory is tight
#     }
# )

Found 19 images
Debug images will be saved to /Users/lianshi/Downloads/bachelor thesis/jupyter/debug/
Marker found at (1604, 917)
Watching zone covers 39.4% of image
Background computed from 19 sampled frames (of 19 total)
Detecting visitors...
Loading model...
Model loaded.

WSCT2946.JPG: no visitor detected (sunny)
WSCT2947.JPG: no visitor detected (sunny)
WSCT2948.JPG: no visitor detected (sunny)
WSCT2949.JPG: no visitor detected (sunny)
WSCT2950.JPG: no visitor detected (sunny)
WSCT2951.JPG: no visitor detected (sunny)
WSCT2952.JPG: no visitor detected (sunny)
WSCT2953.JPG: no visitor detected (sunny)
  WSCT2954.JPG [roi]: skipped — OOD (energy=47.60)
  WSCT2955.JPG [roi]: skipped — OOD (energy=52.94)
  WSCT2955.JPG [outside_roi]: skipped — OOD (energy=117.15)
WSCT2956.JPG: no visitor detected (sunny)
WSCT2957.JPG: no visitor detected (sunny)
  WSCT2958.JPG [outside_roi]: skipped — OOD (energy=35.07)
WSCT2968.JPG: no visitor detected (sunny)
  WSCT2969.JPG [outside_roi]: skipped — 